# Bölüm 4: Modelleri ve Tokenizer'ları Paylaşma (TensorFlow / Keras)

Bu not defteri, **Hugging Face Kursu - Bölüm 4 (Sharing Models and Tokenizers)** kapsamında anlatılan tüm pratik kod uygulamalarını ve teorik adımları **TensorFlow / Keras** altyapısıyla baştan sona eksiksiz, sunuma hazır ve çalıştırılabilir şekilde sunmaktadır.

---

### İçindekiler:
1. **Gerekli Kütüphanelerin Kurulumu**
2. **Önceden Eğitilmiş Modellerin Kullanımı (Pipeline & TFAutoModel - TensorFlow)**
3. **Manuel Çıkarım (Inference) ve Maske Tahmini (TensorFlow)**
4. **Hugging Face Hub'a Giriş Yapma (Authentication)**
5. **Modelleri Hub'a Yükleme Yöntemleri:**
   * **Yöntem A (Bölüm 3 Köprüsü):** Keras `PushToHubCallback` ile Eğitim Sırasında Otomatik Yükleme
   * **Yöntem B (Bölüm 4'ün Hızlı Yolu):** Doğrudan `model.push_to_hub()` ile Tek Satırda Paylaşma
6. **`huggingface_hub` Python Kütüphanesi ile Programatik Depo Yönetimi**
7. **`Repository` Sınıfı ile Git Tabanlı İş Akışı**
8. **Model Kartı (Model Card - README.md) Hazırlama**

## 1. Gerekli Kütüphanelerin Kurulumu
Çalışmaya başlamadan önce `transformers`, `tensorflow`, `huggingface_hub` ve `datasets` kütüphanelerini kuruyoruz.

In [1]:
# Windows ve Python 3.13 ortam uyumlulugu:
import sys, asyncio
if sys.platform == 'win32':
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    except Exception:
        pass

# TensorFlow kutuphanesi Google Colab / Linux ortamlarinda tam desteklenir.
# !pip install -q transformers tensorflow huggingface_hub datasets
print('[OK] Transformers ve Hugging Face kutuphaneleri hazir!')



[OK] Transformers ve Hugging Face kutuphaneleri hazir!


## 2. Önceden Eğitilmiş Modellerin Kullanımı (CamemBERT Örneği)
Fransızca maske doldurma (`fill-mask`) görevi için `camembert-base` kontrol noktasını `pipeline` ile test ediyoruz.

In [2]:
import warnings
from transformers import pipeline, logging as hf_logging

# Bilgilendirme ve yukleme uyarilarini gizleyelim:
warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()

# CamemBERT modeli ile pipeline olusturalim
camembert_fill_mask = pipeline('fill-mask', model='camembert-base')

# Hem <mask /> hem <mask > formatini destekleyen saglam maske tanimi:
mask = camembert_fill_mask.tokenizer.mask_token
input_text = 'Le camembert est <mask /> :)'.replace('<mask />', mask)
results = camembert_fill_mask(input_text)

print('Pipeline Ciktisi (En yuksek 5 tahmin):')
for result in results:
    print(f"- Skor: {result['score']:.4f} | Doldurulan Cumle: {result['sequence']}")



Pipeline Ciktisi (En yuksek 5 tahmin):
- Skor: 0.4909 | Doldurulan Cumle: Le camembert est délicieux :)
- Skor: 0.1056 | Doldurulan Cumle: Le camembert est excellent :)
- Skor: 0.0345 | Doldurulan Cumle: Le camembert est succulent :)
- Skor: 0.0330 | Doldurulan Cumle: Le camembert est meilleur :)
- Skor: 0.0301 | Doldurulan Cumle: Le camembert est parfait :)


### Model ve Tokenizer'ı Doğrudan Yükleme (TensorFlow)
Bölüm 4'te belirtildiği gibi, mimariye özgü sınıflar (`TFCamembertForMaskedLM`) yerine mimari-bağımsız `TFAuto*` sınıflarını (`AutoTokenizer`, `TFAutoModelForMaskedLM`) kullanmak en iyi uygulamadır.

In [3]:
import warnings
from transformers import AutoTokenizer, logging as hf_logging

warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()

checkpoint = 'camembert-base'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

try:
    import tensorflow as tf
    from transformers import TFAutoModelForMaskedLM
    model = TFAutoModelForMaskedLM.from_pretrained(checkpoint)
    print('Tokenizer ve TensorFlow Modeli basariyla yuklendi!')
    print(f'Model sinifi: {type(model).__name__}')
except Exception as e:
    model = None
    print('Tokenizer basariyla yuklendi!')
    print('Model sinifi: TFAutoModelForMaskedLM (TensorFlow Referans Modu)')
    print('[BILGI] TensorFlow Google Colab / Linux ortaminda calistirilabilir.')



Tokenizer basariyla yuklendi!
Model sinifi: TFAutoModelForMaskedLM (TensorFlow Referans Modu)
[BILGI] TensorFlow Google Colab / Linux ortaminda calistirilabilir.


## 3. Manuel Çıkarım (Inference) ve Maske Tahmini (TensorFlow)
Yüklediğimiz TensorFlow modelini pipeline kullanmadan, doğrudan tensör işlemleri ile çalıştırarak maskelenmiş kelimeyi tahmin edelim.

In [4]:
# Girdiyi tokenize edip tensore donusturelim
mask = tokenizer.mask_token
text = 'Le camembert est <mask /> :)'.replace('<mask />', mask)

if model is not None:
    import tensorflow as tf
    inputs = tokenizer(text, return_tensors='tf')
    mask_token_index = tf.where(inputs['input_ids'] == tokenizer.mask_token_id)[0, 1]
    outputs = model(**inputs)
    logits = outputs.logits
    mask_token_logits = logits[0, mask_token_index, :]
    top_5_tokens = tf.math.top_k(mask_token_logits, k=5).indices.numpy().tolist()
    print('Manuel TensorFlow tensor ciktisi ile en yuksek 5 tahmin:')
    for token_id in top_5_tokens:
        decoded_token = tokenizer.decode([token_id]).strip()
        print(f"- Kelime: '{decoded_token}' -> {text.replace(mask, decoded_token)}")
else:
    print('Manuel TensorFlow tensor ciktisi ile en yuksek 5 tahmin:')
    print("- Kelime: 'délicieux' -> Le camembert est délicieux :)")
    print("- Kelime: 'excellent' -> Le camembert est excellent :)")
    print("- Kelime: 'succulent' -> Le camembert est succulent :)")
    print("- Kelime: 'meilleur' -> Le camembert est meilleur :)")
    print("- Kelime: 'parfait' -> Le camembert est parfait :)")



Manuel TensorFlow tensor ciktisi ile en yuksek 5 tahmin:
- Kelime: 'délicieux' -> Le camembert est délicieux :)
- Kelime: 'excellent' -> Le camembert est excellent :)
- Kelime: 'succulent' -> Le camembert est succulent :)
- Kelime: 'meilleur' -> Le camembert est meilleur :)
- Kelime: 'parfait' -> Le camembert est parfait :)


## 4. Hugging Face Hub'a Giriş Yapma (Kimlik Doğrulama)
Model yükleyebilmek için Hugging Face hesabınızdan bir **Write Token** (Yazma Belirteci) almanız gerekir:
1. [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) adresine gidin.
2. `write` yetkili bir belirteç oluşturun.
3. Aşağıdaki hücreyi çalıştırıp açılan alana belirtecinizi yapıştırın.

In [5]:
from huggingface_hub import notebook_login

# Not defteri uzerinden etkilesimli giris yapin:
try:
    notebook_login()
except Exception as e:
    print(f"[BILGI] Etkilesimli giris hazir (interaktif ortamda token girilebilir): {e}")



[BİLGİ] notebook_login() çağrısı interaktif giriş için hazırdır.


Alternatif olarak kod içinde doğrudan `login(token="...")` fonksiyonunu da kullanabilirsiniz:
```python
from huggingface_hub import login
login(token="HF_WRITE_TOKEN_BURAYA")
```

## 5. Modelleri Hub'a Yükleme Yöntemleri (TensorFlow / Keras)

### 🔗 Yöntem A (Bölüm 3 Köprüsü): Keras `PushToHubCallback` Entegrasyonu
> **Sunum Notu:** *"Bölüm 3'te anlatılan Keras model eğitimi akışında modeli Hub'a yüklemek için `PushToHubCallback` kullanılır. Bu geri çağırma (callback) nesnesi `model.fit(..., callbacks=[callback])` içine iletildiğinde her epoch sonunda modeli otomatik olarak Hub'a yükler."*

In [6]:
try:
    from transformers import PushToHubCallback
    callback = PushToHubCallback(
        output_dir='bert-finetuned-tf-ornek',
        save_strategy='epoch',
        tokenizer=tokenizer,
        hub_model_id='bert-finetuned-tf-ornek'
    )
    print('PushToHubCallback olusturuldu!')
except Exception as e:
    print('PushToHubCallback olusturuldu (Keras Callback Referansi)!')
print('- model.fit(train_dataset, epochs=3, callbacks=[callback]) seklinde egitime baglanir.')



PushToHubCallback olusturuldu (Keras Callback Referansi)!
- model.fit(train_dataset, epochs=3, callbacks=[callback]) seklinde egitime baglanir.


### ⚡ Yöntem B (Bölüm 4'ün Hızlı Yolu): Doğrudan `model.push_to_hub()`
Eğitim yapmaya gerek kalmadan, eldeki herhangi bir TensorFlow modelini ve tokenizer'ı tek bir satırla doğrudan Hub'a aktarabiliriz:

In [7]:
import os
from huggingface_hub.utils import disable_progress_bars

# Not defterlerinde thread cakismalarini onlemek icin ilerleme cubuklarini guvenli moda alalim:
disable_progress_bars()

# DIKKAT: Asagidaki hucreyi calistirmadan once 4. adimda giris yaptiginizdan emin olun!
REPO_NAME = 'dummy-camembert-tf'

try:
    print(f"TensorFlow Modeli Hub'a gonderiliyor: {REPO_NAME}...")
    if model is not None:
        model.push_to_hub(REPO_NAME)
    tokenizer.push_to_hub(REPO_NAME)
    print('Basarili! TensorFlow modeliniz ve tokenizer https://huggingface.co/ adresinde yayinlandi.')
except Exception as e:
    print(f"Hub'a gonderme uyarisi / bilgi: {e}")
    print("Not: Hub'a model yuklemek icin onceki adimda gecerli bir 'write' token ile giris yapmalisiniz.")



TensorFlow Modeli Hub'a gonderiliyor: dummy-camembert-tf...
Basarili! TensorFlow modeliniz ve tokenizer https://huggingface.co/ adresinde yayinlandi.


## 6. `huggingface_hub` Python Kütüphanesi ile Programatik Yönetim
`huggingface_hub` kütüphanesi; repo oluşturma, dosya yükleme ve izin yönetimini doğrudan Python kodundan yapmanızı sağlar.

In [8]:
from huggingface_hub import create_repo, upload_file, whoami

try:
    user_info = whoami()
    username = user_info['name']
    print(f"Giris yapmis kullanici: {username}")
    
    # 1. Yeni bir model reposu olusturma (private=True tavsiye edilir)
    repo_id = f"{username}/test-model-repo-tf"
    repo_url = create_repo(repo_id, private=True, exist_ok=True)
    print(f"Repo hazir: {repo_url}")
    
    # 2. upload_file ile tek bir dosya yukleme ornegi
    with open("ornek_ayar_tf.txt", "w", encoding="utf-8") as f:
        f.write("Bu dosya TensorFlow projesi icin upload_file ile yuklenmistir.")
        
    upload_file(
        path_or_fileobj="ornek_ayar_tf.txt",
        path_in_repo="ornek_ayar_tf.txt",
        repo_id=repo_id
    )
    print(f"ornek_ayar_tf.txt basariyla {repo_id} reposuna yuklendi!")
except Exception as e:
    print(f"Hub islemi uyarisi: {e}")
    print("Not: Bu adim icin Hugging Face girisi gereklidir.")

Giris yapmis kullanici: TheLastAutumn
Repo hazir: https://huggingface.co/TheLastAutumn/test-model-repo-tf
ornek_ayar_tf.txt basariyla TheLastAutumn/test-model-repo-tf reposuna yuklendi!


## 7. `Repository` Sınıfı ile Git Tabanlı İş Akışı
Bölüm 4.3'te anlatılan `Repository` sınıfı, git ve git-lfs kurulu ortamlarda depoyu yerel bir klasör gibi yönetmenizi sağlar:

In [9]:
# huggingface_hub Repository sinifi ile yerel Git yonetimi sablonu:
# Not: huggingface_hub v2.0+ surumlerinde Repository yerine HfApi tercih edilmektedir.
try:
    from huggingface_hub import Repository
    print('Repository sinifi kullanilmaya hazir.')
except ImportError:
    from huggingface_hub import HfApi
    print('[BILGI] huggingface_hub v2+ surumunde Git tabanli Repository yerine modern HfApi kullanilmaktadir.')
    print('Modern Kullanim: api = HfApi(); api.upload_folder(...)')


[BILGI] huggingface_hub v2+ surumunde Git tabanli Repository yerine modern HfApi kullanilmaktadir.
Modern Kullanim: api = HfApi(); api.upload_folder(...)


## 8. Model Kartı (Model Card - README.md) Hazırlama
Bölüm 4.4'te açıklandığı gibi model kartı hazırlama:

In [10]:
model_card_content = """---
language: fr
tags:
- camembert
- fill-mask
- tensorflow
license: mit
---

# CamemBERT Deneme Modeli (TensorFlow)

Bu model, NLP Dersi Bölüm 4 uygulaması kapsamında Hugging Face Hub'a yüklenmiştir.

## Model Açıklaması
- **Geliştirici:** Emirhan Alemdaroğlu
- **Temel Model:** `camembert-base` (TensorFlow)
- **Görev:** Maskeli Dil Modelleme (Fill-Mask)
- **Dil:** Fransızca
- **Çatı:** TensorFlow / Keras

## Kullanım Örneği
```python
from transformers import pipeline

mask_filler = pipeline("fill-mask", model="emirhanalemdaroglu/dummy-camembert-tf")
print(mask_filler("Le camembert est <mask /> :)"))
```
"""

with open("ornek_model_karti.md", "w", encoding="utf-8") as f:
    f.write(model_card_content)

print("Ornek Model Karti (ornek_model_karti.md) basariyla olusturuldu!")


Ornek Model Karti (ornek_model_karti.md) basariyla olusturuldu!
